# Lezione 10 — Normaliza e construye secuencias temporales

**Manuale associato:** [manual/10_caso_cripto_normaliza_secuencias.md](../manual/10_caso_cripto_normaliza_secuencias.md)

## Resumen para la lezione

- Con series temporales, separa allenamento e validación **por tiempo**, nunca al azar.
- `pct_change()` normaliza activos con escalas de precio muy distintas.
- `preprocessing.scale()` centra e escala cada columna ya convertida en cambio porcentual.
- `collections.deque(maxlen=SEQ_LEN)` construye ventanas deslizantes de tamaño fijo.

In [ ]:
import pandas as pd

main_df = pd.read_pickle("main_df.pickle")
main_df.head()

In [ ]:
tiempos = sorted(main_df.index.values)
umbral_validacion = tiempos[-int(0.05 * len(tiempos))]

df_validacion = main_df[main_df.index >= umbral_validacion]
df_entrenamiento = main_df[main_df.index < umbral_validacion]

print("Filas de entrenamiento:", len(df_entrenamiento))
print("Filas de validacion:", len(df_validacion))

In [ ]:
from sklearn import preprocessing


def preprocesa_df(df):
    df = df.copy()
    for columna in df.columns:
        if columna not in ("target", "future"):
            df[columna] = df[columna].pct_change()
            df.dropna(inplace=True)
            df[columna] = preprocessing.scale(df[columna].values)

    df.dropna(inplace=True)
    return df.drop(columns=["future"])


df_entrenamiento = preprocesa_df(df_entrenamiento)
df_validacion = preprocesa_df(df_validacion)
df_entrenamiento.head()

In [ ]:
from collections import deque
import numpy as np
import random

SEQ_LEN = 60


def crea_secuencias(df):
    datos_secuenciales = []
    ventana = deque(maxlen=SEQ_LEN)
    columnas_features = [c for c in df.columns if c != "target"]

    for fila in df[columnas_features + ["target"]].values:
        ventana.append(fila[:-1])
        if len(ventana) == SEQ_LEN:
            datos_secuenciales.append([np.array(ventana), fila[-1]])

    random.shuffle(datos_secuenciales)
    return datos_secuenciales


secuencias_entrenamiento = crea_secuencias(df_entrenamiento)
secuencias_validacion = crea_secuencias(df_validacion)

print(f"Secuencias de entrenamiento: {len(secuencias_entrenamiento)}")
print(f"Secuencias de validacion: {len(secuencias_validacion)}")

## Cierre de la lezione

- Comprueba que cada secuencia tenga forma `(60, numero_de_columnas)`.
- En la lezione 11 balanceas estas secuencias e entrenas el modello LSTM final.

In [ ]:
import pickle

with open("secuencias_entrenamiento.pickle", "wb") as f:
    pickle.dump(secuencias_entrenamiento, f)
with open("secuencias_validacion.pickle", "wb") as f:
    pickle.dump(secuencias_validacion, f)